# 04 Explaining A2 (WavLM + layer-weighted head)
A2 averages each of WavLM's 13 layers over time, mixes them with learned weights, then a small MLP. Time is averaged away, so instead of frames we look at layers and at silence.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from audio_df.data.manifests import load_manifest
from audio_df.helpers.eval_audio import trimmed_manifest
from audio_df.helpers.metrics import compute_eer
from audio_df.helpers.ssl_cache import cached_pooled
from audio_df.modules.ssl_detector import SSLDetector

CACHE = Path("../data/cache").resolve()  # absolute, so cache identities match the runtime's
art = sorted(Path("../artifacts/a2").glob("*/COMPLETE"))[-1].parent
prep = art / "preprocessing.json"
train_audio = json.loads(prep.read_text())["train"] if prep.exists() else "raw"
a2 = SSLDetector.load(art, device="cpu")
print(art.name, "| trained on", train_audio, "audio")

def eer(scores, labels):
    return round(100 * compute_eer(scores, labels, higher_is_spoof=False)[0], 2)

def feats(df, split):  # cached [clips, 13 layers, 768], no WavLM run when cached
    return cached_pooled(df, a2.encoder, CACHE / "wavlm-base-plus/pooled", split).astype(np.float32)

train, dev = load_manifest("../data/manifests/train.csv", ".."), load_manifest("../data/manifests/dev.csv", "..")
if train_audio == "trimmed":
    train = trimmed_manifest(train, CACHE, "train")
sets = {"train": train, "dev raw": dev, "dev trimmed": trimmed_manifest(dev, CACHE, "dev")}
X = {k: feats(v, "train" if k == "train" else "dev") for k, v in sets.items()}
y = {k: v.label.to_numpy() for k, v in sets.items()}

## 1 Learned layer weights

In [ ]:
w = a2.head.layer_weights().detach().numpy()
plt.figure(figsize=(7, 2.5))
plt.bar(range(13), w)
plt.axhline(1 / 13, color="grey", ls=":", label="uniform")
plt.xlabel("WavLM layer (0 = CNN output)")
plt.ylabel("weight")
plt.legend()
plt.show()

Almost uniform (0.06–0.09), slightly more on early layers. The head barely picks a layer, it mostly averages all 13.

## 2 What does each layer know?
A linear probe per layer, trained on train, scored on raw and trimmed dev. Dashed lines = A2 itself.

In [ ]:
def probe(layer):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1, class_weight="balanced"))
    clf.fit(X["train"][:, layer], y["train"] == "bonafide")  # positive class = bona fide, like A2
    return {k: eer(clf.decision_function(X[k][:, layer]), y[k]) for k in ["dev raw", "dev trimmed"]}

def a2_scores(x):
    with torch.inference_mode():
        return a2.head(torch.from_numpy(x)).numpy()

layers = pd.DataFrame({l: probe(l) for l in range(13)}).T
full = {k: eer(a2_scores(X[k]), y[k]) for k in ["dev raw", "dev trimmed"]}
ax = layers.plot(marker="o", figsize=(7, 3), ylabel="EER %", xlabel="WavLM layer")
for k, c in zip(full, ["C0", "C1"]):
    ax.axhline(full[k], color=c, ls="--")
plt.show()
layers.T.assign(A2=pd.Series(full))

On raw audio every layer is under 4%, so every layer can find the silence. On trimmed audio only the middle layers hold up: layer 4–5 alone get 5.5–6%, as good as A2 itself (5.5%). The CNN output (layer 0) and the last layers (8–12) drop to 16–19%. Layer 12 is what D2/D3 use, one of the worst after trimming.

## 3 Silence test
Same A2, same dev clips, raw vs trimmed audio.

In [ ]:
s = {k: a2_scores(X[k]) for k in ["dev raw", "dev trimmed"]}
shift = s["dev trimmed"] - s["dev raw"]
print(pd.DataFrame({k: pd.Series(v).groupby(y[k]).mean() for k, v in s.items()}).round(2))
plt.figure(figsize=(7, 2.5))
for lab, color in [("bonafide", "tab:green"), ("spoof", "tab:orange")]:
    plt.hist(shift[y["dev raw"] == lab], bins=80, density=True, histtype="step", color=color, label=lab)
plt.axvline(0, color="grey", lw=0.5)
plt.xlabel("score change after trimming (- = more spoof)")
plt.legend()
plt.show()

Trimming moves bona fide clips about 11 points towards spoof while spoof barely moves. A2 was counting silence as bona fide evidence; without it, bona fide drops to a mean of -0.9, close to the spoof side.